In [1]:
import torch
import time
import pandas as pd
import os
import torch.nn.utils.prune as prune

from torchvision.models import mobilenet_v2

In [2]:
model = mobilenet_v2(weights="DEFAULT")
model.eval()

print("Model loaded.")

Model loaded.


In [3]:
quantized_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

print("Quantization applied.")

Quantization applied.


C:\Users\user\AppData\Local\Temp\ipykernel_14116\1421447565.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


In [4]:
for module in quantized_model.modules():
    if isinstance(module, torch.nn.Conv2d):
        prune.ln_structured(
            module,
            name="weight",
            amount=0.2,
            n=2,
            dim=0
        )

print("20% structured pruning applied.")

20% structured pruning applied.


In [5]:
for module in quantized_model.modules():
    if isinstance(module, torch.nn.Conv2d):
        try:
            prune.remove(module, "weight")
        except:
            pass

print("Pruning made permanent.")

Pruning made permanent.


In [6]:
def benchmark(model, device, input_tensor, runs=100):

    model = model.to(device)
    input_tensor = input_tensor.to(device)

    for _ in range(10):
        with torch.no_grad():
            _ = model(input_tensor)

    if device == "cuda":
        torch.cuda.synchronize()

    start = time.time()

    for _ in range(runs):
        with torch.no_grad():
            _ = model(input_tensor)

    if device == "cuda":
        torch.cuda.synchronize()

    end = time.time()

    avg_latency = (end - start) / runs

    return avg_latency

In [7]:
input_tensor = torch.randn(1, 3, 224, 224)

In [8]:
cpu_latency = benchmark(
    quantized_model,
    "cpu",
    input_tensor
)

print(f"CPU Average Latency: {cpu_latency:.6f} seconds")

CPU Average Latency: 0.065440 seconds


In [9]:
torch.save(
    quantized_model.state_dict(),
    "quantize_then_prune_model.pth"
)

size_mb = os.path.getsize(
    "quantize_then_prune_model.pth"
) / (1024 * 1024)

print(f"Model size: {size_mb:.2f} MB")

Model size: 9.94 MB
